# Reddit Pain-Point Finder

This script executes functions at `/lib/reddit.py` to get a report of pain points given a subreddit.


## 01. Setup


In [47]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

from dotenv import load_dotenv
print("env loaded:", load_dotenv("../.env", override=True))

from lib import reddit as Reddit
from lib import tokens as Tok
from lib import llm as LLM

from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
import time
import re

r = Reddit.get_client()
print(Reddit.check_auth(r))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env loaded: True
{'read_only': True, 'subscribers': 86541, 'limits': {'remaining': 999, 'used': 1}}


## 02. Discovery


In [48]:
SEEDS = [
    "Collectr",
    "Ludex",
    "PriceCharting",
    "Card Ladder",
    "Pokellector",
    "TCGPlayer",
    "Whatnot",
    "PSA grading",
]

rows = Reddit.discover_subreddits(r, SEEDS, limit_per_term=100)

df = pd.DataFrame(rows)
df = df[df.hits >= 3].sort_values(["distinct_terms", "hits"], ascending=False)
print(df[["subreddit", "hits", "distinct_terms", "terms"]].head(20).to_string())

  'Collectr' -> 100 results
  'Ludex' -> 100 results
  'PriceCharting' -> 100 results
  'Card Ladder' -> 100 results
  'Pokellector' -> 100 results
  'TCGPlayer' -> 100 results
  'Whatnot' -> 100 results
  'PSA grading' -> 100 results
                subreddit  hits  distinct_terms                                                                    terms
0              PokemonTCG    89               6  [Collectr, PSA grading, Pokellector, PriceCharting, TCGPlayer, Whatnot]
12            sportscards    13               6   [Card Ladder, Ludex, PSA grading, Pokellector, PriceCharting, Whatnot]
3           PokeInvesting    37               5             [Card Ladder, Collectr, PSA grading, PriceCharting, Whatnot]
15  pokemoncardcollectors     9               5             [Collectr, PSA grading, Pokellector, PriceCharting, Whatnot]
9           baseballcards    15               4                               [Card Ladder, Ludex, PSA grading, Whatnot]
13                    mtg    12        

## 03. Configuration


In [49]:
IDEA_SLUG = "pokemon-collector-onboarding"

# PLACEHOLDER - replace from discovery output. My guesses have been wrong
# three times in a row; run cell 2 first.
CORE_SUBS = [
    "PokemonTCG",
    "pkmntcgcollections",
    "PokeInvesting",
]

# No wholesale pass this time - these subs are far too big for it.
SMALL_SUBS = []

# What beginners struggle WITH
DOMAIN_NOUNS = [
    "fake cards",
    "grading",
    "PSA submission",
    "card condition",
    "sealed product",
    "collection tracking",
    "card values",
]

# How beginners describe themselves
BEGINNER_NOUNS = [
    "just started collecting",
    "new collector",
    "getting back into",
    "where to start",
]

# The tools they already use or abandon
COMPETITORS = [
    "Collectr", "Ludex", "PriceCharting", "Card Ladder",
    "Pokellector", "TCGplayer", "Whatnot", "eBay",
]

ALL_NOUNS = DOMAIN_NOUNS + BEGINNER_NOUNS

# 1. Pain grammar x nouns
QUERIES  = [f'"{n}" "{p}"' for p in Reddit.PAIN_PHRASES[:5] for n in ALL_NOUNS]

# 2. Loss language - the vertical's version of workaround hunting.
#    Money lost is the signal that separates confusion from real pain.
QUERIES += [f'"{n}" {t}' for n in DOMAIN_NOUNS
            for t in ["ripped off", "overpaid", "regret", "spreadsheet"]]

# 3. Tool complaints
QUERIES += [f'{c} {t}' for c in COMPETITORS
            for t in ["alternative", "inaccurate", "hate"]]

cfg = Reddit.HarvestConfig(
    subreddits=CORE_SUBS,
    queries=QUERIES,
    time_filters=["year"],     # market moves fast; 2021 pain != today's
    sorts=["relevance"],
    limit_per_search=60,
    comments_per_post=40,
)

n_search = len(CORE_SUBS) * len(QUERIES)
print(f"{len(QUERIES)} queries x {len(CORE_SUBS)} subs = {n_search} searches")

107 queries x 3 subs = 321 searches


## 04. Harvest core subs


In [50]:
subs, comments = Reddit.harvest(r, cfg)
print(f"{len(subs)} submissions, {len(comments)} comments from core subs")

   r/PokemonTCG [relevance/year] '"fake cards" "there's no way to"' -> +1 new 1 raw
   r/PokemonTCG [relevance/year] '"fake cards" "is there a way to"' -> +2 new 2 raw
   r/PokemonTCG [relevance/year] '"grading" "is there a way to"' -> +6 new 6 raw
   r/PokemonTCG [relevance/year] '"new collector" "is there a way to"' -> +1 new 1 raw
   r/PokemonTCG [relevance/year] '"getting back into" "is there a way to"' -> +1 new 1 raw
   r/PokemonTCG [relevance/year] '"fake cards" ripped off' -> +2 new 2 raw
   r/PokemonTCG [relevance/year] '"fake cards" regret' -> +1 new 1 raw
   r/PokemonTCG [relevance/year] '"grading" ripped off' -> +37 new 37 raw
   r/PokemonTCG [relevance/year] '"grading" overpaid' -> +3 new 3 raw
   r/PokemonTCG [relevance/year] '"grading" regret' -> +26 new 26 raw
   r/PokemonTCG [relevance/year] '"grading" spreadsheet' -> +7 new 7 raw
   r/PokemonTCG [relevance/year] '"sealed product" ripped off' -> +11 new 11 raw
   r/PokemonTCG [relevance/year] '"sealed product" regret' 

## 05. Harvest small subs


In [51]:
# Small subs: whole listing, no queries
listing = []
for s in SMALL_SUBS:
    batch = Reddit.fetch_listing(r, s, sort="top", time_filter="all", limit=500)
    print(f"  r/{s}: {len(batch)} posts")
    listing += batch

seen = {x["id"] for x in subs}
extra = [x for x in listing if x["id"] not in seen]

extra_comments = []
for i, s in enumerate(extra, 1):
    extra_comments += Reddit.fetch_comments(r, s["id"], limit=40)
    if i % 25 == 0:
        print(f"  {i}/{len(extra)} -> {len(extra_comments)} comments")
print(f"{len(extra)} posts, {len(extra_comments)} comments from small subs")

0 posts, 0 comments from small subs


## 06. Save


In [52]:
run_id = f"{IDEA_SLUG}-{int(time.time())}"
path = Reddit.save_json(subs + comments + extra + extra_comments,
                        f"../runs/{run_id}/raw.jsonl")
print(f"saved -> {path}")

saved -> ../runs/pokemon-collector-onboarding-1787782270/raw.jsonl


## 07. Load and score


In [53]:
records = Reddit.load_json(path)
print(f"{len(records)} records from {path}")

PAIN_RE = [re.compile(p, re.I) for p in [
    r"\bripped off\b", r"\bover ?paid\b", r"\bwasted?\b", r"\bfake\b",
    r"\bcounterfeit\b", r"\bproxy\b", r"\bhow do i (know|tell|check)\b",
    r"\bis this (worth|real|legit)\b", r"\bshould i (grade|buy|sell)\b",
    r"\bspreadsheet\b", r"\bmanually\b", r"\bkeep track\b", r"\btracking\b",
    r"\bregret\b", r"\bmistake\b", r"\blost \$?\d", r"\bwish i (had|knew)\b",
    r"\bno idea (what|how|where)\b", r"\bwhere do i (start|begin)\b",
    r"\bgraded? (wrong|badly)\b", r"\bdidn'?t know\b",
]]

NEWBIE_RE = [re.compile(p, re.I) for p in [
    r"\bjust (started|got into|getting into)\b", r"\bnew to (collecting|the hobby|tcg)\b",
    r"\bbeginner\b", r"\breturning (to the hobby|collector)\b",
    r"\bfirst (time|purchase|order|slab)\b", r"\bgetting back into\b",
]]

TOOL_RE = re.compile("|".join(SEEDS), re.I)

def score(rec):
    text = f"{rec.get('title') or ''} {rec.get('body') or ''}"
    if len(text) < 80:
        return 0
    s  = sum(bool(p.search(text)) for p in PAIN_RE)
    s += 2 * sum(bool(p.search(text)) for p in NEWBIE_RE)
    s += 2 * bool(TOOL_RE.search(text))
    return s

for rec in records:
    rec["prefilter"] = score(rec)

print(pd.Series([x["prefilter"] for x in records]).value_counts().sort_index())

10022 records from ../runs/pokemon-collector-onboarding-1787782270/raw.jsonl
0    9286
1     351
2     311
3      49
4      17
5       6
6       1
8       1
Name: count, dtype: int64


## 08. Where the signal lives


In [54]:
d = pd.DataFrame([{"sub": x["subreddit"], "score": x["prefilter"]} for x in records])
print(pd.crosstab(d["sub"], d["score"] >= 3))
print("\nscore >= 2 by sub:")
print(d[d.score >= 2]["sub"].value_counts())
print("\ndistinct authors:", len({x.get("author") for x in records}))

score               False  True 
sub                             
PokeInvesting        4217     26
PokemonTCG           4823     48
pkmntcgcollections    908      0

score >= 2 by sub:
sub
PokemonTCG            220
PokeInvesting         160
pkmntcgcollections      5
Name: count, dtype: int64

distinct authors: 5620


## 09. Client, and verify the two speed factors


In [55]:
client = LLM.LocalLLM(model="qwen3:14b")
print("endpoint:", client.base_url)
print("models:", LLM.health_check(client.base_url))

# Is thinking mode actually off? A <think> block here means you're paying 6x.
import requests
resp = requests.post(f"{client.base_url}/chat/completions", json={
    "model": "qwen3:14b", "temperature": 0,
    "messages": [{"role": "system", "content": LLM.SYSTEM},
                 {"role": "user", "content": "we track class signups in a spreadsheet every week"}],
}, timeout=180)
print(resp.json()["choices"][0]["message"]["content"][:300])

endpoint: http://100.69.132.64:11434/v1
models: ['qwen3:14b']
{"pain":false,"problem":null,"workaround":null,"tools":[],"experience":null,"money":null,"severity":null}


## 10. Extract the top tier


In [56]:
top = sorted([x for x in records if x["prefilter"] >= 3],
             key=lambda x: -x["prefilter"])
sample = [Tok.item_text(x, 1200) for x in top]

by_worker, by_status = Counter(), Counter()
bar = tqdm(total=len(sample), desc="extract", unit="item")

def progress(done, total, status, worker):
    by_worker[worker] += 1
    by_status[status.split(":")[0]] += 1
    bar.update(1)
    bar.set_postfix(dict(by_status))

recs, statuses = client.extract_many(sample, workers=4, on_progress=progress)
bar.close()

print("per-worker:", dict(by_worker))
print(Counter(statuses))
print("pain rate:", sum(x["pain"] for x in recs if x) / len(recs))

extract: 100%|██████████| 74/74 [06:31<00:00,  5.29s/item, ok=74]

per-worker: {'w_0': 17, 'w_2': 18, 'w_1': 20, 'w_3': 19}
Counter({'ok': 74})
pain rate: 0.43243243243243246


In [58]:
for src, rec in zip(top, recs):
    if rec and rec["pain"]:
        print("─" * 70)
        print(f"r/{src['subreddit']} | sev {rec['severity']} | money {rec['money']}")
        print("PROB :", rec["problem"])
        print("WORK :", rec["workaround"])
        print("TOOLS:", rec["tools"], "| ROLE:", rec["experience"])
        print("LINK :", src["permalink"])

──────────────────────────────────────────────────────────────────────
r/PokemonTCG | sev 2 | money False
PROB : Difficulty deciding whether to sell high-value cards to a shop or list them individually to avoid being underpaid for the rest of the collection
WORK : Considering selling high-value cards to a shop and bulk-selling the rest
TOOLS: ['tcgplayer'] | ROLE: unknown
LINK : https://reddit.com/r/PokemonTCG/comments/1ngtw80/sell_to_shop_or_split_advice_to_not_get_taken/
──────────────────────────────────────────────────────────────────────
r/PokemonTCG | sev 2 | money True
PROB : Vendor added an extra $50 to a trade for a sealed promo card, claiming profit margin on the traded card.
WORK : None
TOOLS: [] | ROLE: new
LINK : https://reddit.com/r/PokemonTCG/comments/1tutds2/went_to_my_first_card_show_last_weekend_and_not/
──────────────────────────────────────────────────────────────────────
r/PokemonTCG | sev 3 | money False
PROB : Managing an increasingly complex spreadsheet with mor

In [46]:
# Where do the hits come from?
hits = [(s, x) for s, x in zip(top, recs) if x and x["pain"]]
print(Counter(s["subreddit"] for s, _ in hits))
print("distinct authors:", len({s.get("author") for s, _ in hits}))

# And read what it REJECTED at the very top of the score range
for src, rec in list(zip(top, recs))[:10]:
    if rec and not rec["pain"]:
        print("─" * 70)
        print(f"score {src['prefilter']} | r/{src['subreddit']}")
        print(Tok.item_text(src, 350).replace("\n", " "))

Counter({'mindbody': 11, 'FitnessStudioOwner': 4, 'personaltraining': 4, 'GymOwnerNetwork': 4, 'YogaTeachers': 4, 'gymowner': 4})
distinct authors: 28
──────────────────────────────────────────────────────────────────────
score 6 | r/gymowner
Hey! We run a smaller gym (about 220 members) and went with Gymwyse after trying a couple of the bigger names. Honestly, we needed something that wasn't going to require a whole IT department to figure out.  The online booking has been smooth for us. Members can book classes through the app without texting us at midnight, which was happening way to [...]
──────────────────────────────────────────────────────────────────────
score 6 | r/mindbody
I left for Arketa and just want to share my experience so NO ONE has to go thru this. We are leaving after a year for Walla, which has been excellent so far (a week in though).  I've worked in technology for 17 years. In all that time, I have never seen a platform fail as comprehensively as Arketa has faile

## 11. Inspect


In [36]:
for src, rec in zip(top, recs):
    if rec and rec["pain"]:
        print("─" * 70)
        print("TEXT :", Tok.item_text(src, 300).replace("\n", " "))
        print("PROB :", rec["problem"])
        print("WORK :", rec["workaround"], "| MONEY:", rec["money"], "| SEV:", rec["severity"])
        print("ROLE :", rec["role"], "| TOOLS:", rec["tools"])
        print("LINK :", src["permalink"])

──────────────────────────────────────────────────────────────────────
TEXT : Hey! We run a smaller gym (about 220 members) and went with Gymwyse after trying a couple of the bigger names. Honestly, we needed something that wasn't going to require a whole IT department to figure out.  The online booking has been smooth for us. Members can book classes through the app without  [...]
PROB : Cash flow timing creates recurring financial stress even when profit is achieved
WORK : Adjusting payment terms (stronger first payments, shorter plans, incentives for upfront payments) | MONEY: True | SEV: 3
ROLE : Coach | TOOLS: []
LINK : https://reddit.com/r/gymowner/comments/1t9bopx/what_crm_are_you_using_for_your_gym_looking_for/ol1lfcu/
